In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ! pip install datasets transformers sacrebleu torch sentencepiece transformers[sentencepiece]
! pip install --upgrade datasets transformers sacrebleu transformers[sentencepiece]
!pip install torch==2.3.0 torchtext==0.18.0 torchaudio==2.3.0 torchvision==0.18.0
# !pip install pyarrow==14.0.1
%pip install evaluate

In [ ]:
import os
import random
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import AdamW
import datasets
import sys
import numpy as np

In [ ]:
df = pd.read_csv('train.csv')

df = df.drop_duplicates()
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

erroneous_sources = list(df['source sentence'].values)
correct_targets = list(df['target sentence'].values)

len(erroneous_sources) == len(correct_targets), len(erroneous_sources)

In [ ]:
train_df = df
test_df = pd.read_csv('test.csv')

test_df = test_df.drop_duplicates()
test_df = test_df.replace([np.inf, -np.inf], np.nan)
test_df = test_df.dropna()

In [ ]:
train_sources = list(train_df['source sentence'].values)
test_sources = list(test_df['source sentence'].values)
train_targets = list(train_df['target sentence'].values)
test_targets = list(test_df['target sentence'].values)

len(train_sources), len(test_sources)

In [ ]:
from transformers import T5Tokenizer, T5Model

tokenizer = AutoTokenizer.from_pretrained("t5-small")

In [ ]:
train_sources_encoding = tokenizer(train_sources, truncation=True, padding=True)
train_targets_encoding = tokenizer(train_targets, truncation=True, padding=True)

test_sources_encoding = tokenizer(test_sources, truncation=True, padding=True)
test_targets_encoding = tokenizer(test_targets, truncation=True, padding=True)

In [ ]:
len(train_targets_encoding['input_ids']), len(test_sources_encoding['input_ids'])

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device

In [ ]:
class LoadDataset(torch.utils.data.Dataset):
    def __init__(self, source_encodings, mask_encodings, targets_encodings):
        self.source_encodings = source_encodings
        self.mask_encodings = mask_encodings
        self.targets_encodings = targets_encodings

    def __getitem__(self, idx):
        input_ids = torch.tensor(self.source_encodings[idx]).squeeze()
        attention_mask = torch.tensor(self.mask_encodings[idx]).squeeze()
        target_ids = torch.tensor(self.targets_encodings[idx]).squeeze()
        return input_ids, attention_mask, target_ids

    def __len__(self):
        return len(self.source_encodings)

In [ ]:
train_dataset = LoadDataset(train_sources_encoding['input_ids'], train_sources_encoding['attention_mask'], train_targets_encoding['input_ids'])
test_dataset = LoadDataset(test_sources_encoding['input_ids'], test_sources_encoding['attention_mask'], test_targets_encoding['input_ids'])

In [ ]:
train_dataset.__len__(), test_dataset.__len__()

In [ ]:
train_dataset.__getitem__(0)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
for source, mask, target in train_loader:
    print(source.shape)
    print(mask.shape)
    print(target.shape)
    break

In [ ]:
model_checkpoint = 't5-small'  # 60.51 T5Small

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
# model.to(device)
print()
print(model.num_parameters())

In [ ]:
model_checkpoint = 't5-small'

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model.to(device)
print()

In [ ]:
optim = torch.optim.AdamW(model.parameters(), lr=5e-5)
N_EPOCHS = 10
epoch = 0
loss = 10e9

PATH = '/kaggle/working/T5_WMT14_V2.pth'
BEST_PATH = '/kaggle/working/T5_WMT14_BEST_PATH_V2.pth'

In [ ]:
print("incorporating model checkpoint")

if os.path.exists(PATH):
    checkpoint = torch.load(PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

print("incorporated model checkpoint")

In [ ]:
print("incorporating best model checkpoint")

if os.path.exists(BEST_PATH):
    checkpoint = torch.load(BEST_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

print("incorporated best model checkpoint")

In [ ]:
print("transfering the checkpoint")

P_PATH = '/kaggle/input/t5_bookcorpus_wmt14_checkpoint/pytorch/v1/1/T5_WMT14_V1.pth'

if os.path.exists(P_PATH):
    checkpoint = torch.load(P_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

epoch += 1

print("checkpoint transfered")

In [ ]:
print("training has started")

for epoch in range(epoch, N_EPOCHS):
    print(f"Epoch = {epoch}")
    epoch_loss = 0
    model.train()

    for (input_ids, attention_mask, target_ids) in tqdm(train_loader):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        target_ids = target_ids.to(device)

        optim.zero_grad()
        predictions = model(input_ids=input_ids, attention_mask=attention_mask, labels=target_ids)
        loss = predictions[0]
        loss.backward()
        epoch_loss += loss.item()
        optim.step()

    epoch_loss = epoch_loss/len(train_loader)
    print(f"Loss = {epoch_loss}")

    if epoch_loss < loss:
        loss = epoch_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'loss': loss,
        }, BEST_PATH)
        best_epoch = epoch
        print(f"{'-'*50}\nModel Saved at {BEST_PATH}\n{'-'*50}\n")

    else:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'loss': loss,
        }, PATH)
        print(f"{'-'*50}\nModel Saved at {PATH}\n{'-'*50}\n")

In [ ]:
print("incorporating latest model checkpoint")

PATH = '/kaggle/input/t5-wmt19-checkpoint/pytorch/evaluation/1/T5_WMT19_BEST_PATH_V3.pth'

if os.path.exists(PATH):
    checkpoint = torch.load(PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']

print("incorporated latest model checkpoint")

In [ ]:
import numpy as np

# Function to calculate perplexity
def calculate_ppl(model, test_loader, device):
    model.eval()
    test_loss = 0

    with torch.no_grad():
        for (input_ids, attention_mask, target_ids) in tqdm(test_loader):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            target_ids = target_ids.to(device)

            # Get the model's predictions
            predictions = model(input_ids=input_ids, attention_mask=attention_mask, labels=target_ids)
            loss = predictions[0]
            test_loss += loss.item()

    # Average loss over all test samples
    test_loss = test_loss / len(test_loader)

    # Calculate Perplexity (PPL) using np.exp
    ppl = np.exp(test_loss)
    
    return ppl

# After training loop, compute PPL on the test set
ppl_score = calculate_ppl(model, test_loader, device)
print(f"Perplexity Score on the Test Set: {ppl_score}")

# **Calculate SacreBLEU, WER & BERTScore**

In [ ]:
class CheckInferenceData(torch.utils.data.Dataset):
    def __init__(self, source_encodings, mask_encodings):
        self.source_encodings = source_encodings
        self.mask_encodings = mask_encodings

    def __getitem__(self, idx):
        input_ids = torch.tensor(self.source_encodings[idx]).squeeze()
        attention_mask = torch.tensor(self.mask_encodings[idx]).squeeze()
        return input_ids, attention_mask
    def __len__(self):
        return len(self.source_encodings)

In [ ]:
test_text_source = ["Mental health is a crucial aspect of overall well-being, affecting how we think"]
test_text_source_encoding = tokenizer(test_text_source, truncation=True, padding=True)

demo_text = CheckInferenceData(test_text_source_encoding['input_ids'], test_text_source_encoding['attention_mask'])
demo_loader = DataLoader(demo_text, batch_size=1, shuffle=False)

for input_ids_demo, attention_mask_demo in demo_loader:
    
    input_ids_demo = input_ids_demo.to(device)
    attention_mask_demo = attention_mask_demo.to(device)
    predictions_demo = model.generate(input_ids=input_ids_demo, attention_mask=attention_mask_demo)
    demo_prediction = [tokenizer.decode(token, skip_special_tokens=True) for token in predictions_demo]

In [ ]:
demo_prediction

In [ ]:
len(test_text_source_encoding['input_ids'][0])

In [ ]:
!pip install sacrebleu
!pip install evaluate
!pip install bert_score evaluate
!pip install torchmetrics
!pip install rouge_score

In [ ]:
import datasets
from evaluate import load
import nltk
from nltk.translate.bleu_score import sentence_bleu
import sacrebleu
import evaluate


sacrebleu = evaluate.load("sacrebleu")
bertscore = load("bertscore")
bleu = evaluate.load("bleu")
rouge = evaluate.load('rouge')

In [ ]:
def calculate_wer(reference, candidate):
    reference = reference.split()
    candidate = candidate.split()

    # Create a matrix to store distances
    distance_matrix = np.zeros((len(reference) + 1) * (len(candidate) + 1), dtype=int).reshape((len(reference) + 1, len(candidate) + 1))

    for i in range(len(reference) + 1):
        for j in range(len(candidate) + 1):
            if i == 0:
                distance_matrix[i][j] = j
            elif j == 0:
                distance_matrix[i][j] = i
            elif reference[i - 1] == candidate[j - 1]:
                distance_matrix[i][j] = distance_matrix[i - 1][j - 1]
            else:
                distance_matrix[i][j] = 1 + min(distance_matrix[i][j - 1],      # Insert
                                               distance_matrix[i - 1][j],      # Remove
                                               distance_matrix[i - 1][j - 1])  # Replace

    return distance_matrix[len(reference)][len(candidate)]

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
import re

def autoregressive_decoding(sentence):

    if isinstance(sentence, list):
        sentence = ' '.join(sentence)
    
    # Remove subword markers "##" by merging with the preceding word
    sentence = sentence.replace(" ##", "")

    # Remove spaces before punctuation (comma, period, etc.)
    sentence = re.sub(r'\s+([.,!?;:])', r'\1', sentence)

    # Remove spaces around single and double quotes
    sentence = re.sub(r"\s*(['\"])\s*", r'\1', sentence)

    # Remove spaces around apostrophes in contractions
    sentence = re.sub(r"\s*'\s*(\w+)", r"'\1", sentence)

    # Replace multiple spaces with a single space
    sentence = re.sub(r'\s+', ' ', sentence).strip()

    # Remove spaces after opening and before closing parentheses/brackets
    sentence = re.sub(r'\s*\(\s*', '(', sentence)
    sentence = re.sub(r'\s*\)\s*', ')', sentence)

    # Ensure proper ellipsis formatting (already in the original code)
    sentence = re.sub(r'\s*\.\s*\.\s*\.\s*', '...', sentence)

    # Remove spaces around hyphens or dashes
    sentence = re.sub(r'\s*-\s*', '-', sentence)

    # Place punctuation inside quotation marks (handle most common cases)
    sentence = re.sub(r'([.,!?;:])\s*([\'"])', r'\2\1', sentence)

    # Remove spaces around "@" for email addresses and Twitter handles
    sentence = re.sub(r'\s*@\s*', '@', sentence)

    # Remove spaces around slashes (e.g., in URLs or subreddit names)
    sentence = re.sub(r'\s*/\s*', '/', sentence)

    # Remove spaces around currency symbols and numbers
    sentence = re.sub(r'\s*([$€£])\s*([0-9]+)', r' \1\2', sentence)
    sentence = re.sub(r'([0-9]+)\s*([,])\s*([0-9]+)', r'\1\2\3', sentence)
    sentence = re.sub(r'([0-9]+)\s*([%])', r'\1\2', sentence)

    # Remove spaces within URLs
    sentence = re.sub(r'\s*(http://|https://|www\.)\s*', r'\1', sentence)

    # Remove spaces in dates and times
    sentence = re.sub(r'\s*/\s*', '/', sentence)  # For dates with slashes
    sentence = re.sub(r'\s*:\s*', ':', sentence)  # For times or other similar cases

    # Remove spaces around mathematical operators
    sentence = re.sub(r'\s*([+\-*/=])\s*', r'\1', sentence)

    # Remove spaces around file extensions
    sentence = re.sub(r'\s+(\.[a-zA-Z0-9]+)', r'\1', sentence)

    # Remove spaces around code-like markers (assuming `code` is delimited by backticks)
    sentence = re.sub(r'\s*(```)\s*', r'\1', sentence)

    # --- New logic for removing spaces after dots ---

    # 1. Remove spaces after a dot when followed by a number (for decimals, currency, dates)
    sentence = re.sub(r'([0-9])\.\s+([0-9])', r'\1.\2', sentence)

    # 2. Remove spaces after a dot when followed by a letter (for abbreviations like U.S.A)
    sentence = re.sub(r'([A-Za-z])\.\s+([A-Za-z])', r'\1.\2', sentence)

    # 3. Remove spaces after a dot when followed by punctuation (e.g., ". , ;")
    sentence = re.sub(r'\.\s+([.,!?;:])', r'.\1', sentence)

    # Ensure there's a space before an opening parenthesis if preceded by a word
    sentence = re.sub(r'(\w)\(', r'\1 (', sentence)

    # Ensure there's a space after a closing parenthesis if followed by a word
    sentence = re.sub(r'\)(\w)', r') \1', sentence)

    # Ensure there's a space after a closing parenthesis if followed by any symbol (%, $, etc.)
    sentence = re.sub(r'\)([^\w\s])', r') \1', sentence)

    # Remove spaces between closing parenthesis and punctuation
    sentence = re.sub(r'\)\s+([,.;!?])', r')\1', sentence)

    # 4. Handle cases where a dot is followed by multiple dots but not an ellipsis (e.g., "U.S. ...")
    sentence = re.sub(r'\.\s+\.', r'..', sentence)

    # Add spaces around single or double quotes for 'protected' words
    sentence = re.sub(r"(\S)(['\"])(\w+)(['\"])(\S)", r"\1 \2\3\4 \5", sentence)

    return sentence

In [ ]:
print("evaluation begins")

model.eval()

total_bleu_score = 0
total_sacre_bleu_score = 0
total_wer_score = 0
total_precision = 0
total_recall = 0
total_f1 = 0

# Initialize variables for Rouge scores
total_rouge1_score = 0
total_rouge2_score = 0
total_rougeL_score = 0

# Create lists to store individual scores for each sentence
bleu_scores = []
sacre_bleu_scores = []
wer_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

# Create list for storing DAE generated output and target text
dae_generated_output = []
target_text_list = []

# Create lists to store Rouge scores for each sentence
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for (input_ids, attention_mask, target_ids) in tqdm(test_loader):
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    target_ids = target_ids.to(device)

    predictions = model.generate(input_ids=input_ids, attention_mask=attention_mask)
    # predictions = model.generate(input_ids=input_ids, attention_mask=attention_mask, labels=target_ids)
    # print(predictions.shape, target_ids.shape)

    target_sentence = [tokenizer.decode(token, skip_special_tokens=True) for token in target_ids]
    output_text = [tokenizer.decode(token, skip_special_tokens=True) for token in predictions]


    target_sentence = ' '.join(target_sentence)
    output_text = ' '.join(output_text)
    
#     output_text = autoregressive_decoding(transformer_output)
    
#     target_sentence = target_sentence.lower()

    # Calculate BLEU score using SacreBLEU
    temp_bleu_score = bleu.compute(predictions=[output_text], references=[[target_sentence]], max_order=2)
    bleu_score = temp_bleu_score["bleu"]

    if bleu_score < 1 or bleu_score < 1.0:
        dae_generated_output.append(output_text)
        target_text_list.append(target_sentence + " BLEU SCORE" + str(bleu_score))
#         print(f"\nDAE output: {output_text}\ntarget sentence: {target_sentence}\n")
#         print(f"\nBLEU Score: {bleu_score}\n")

    # Calculate Rouge scores
    temp_rouge_score = rouge.compute(predictions=[output_text], references=[target_sentence])
    rouge1_score = temp_rouge_score["rouge1"]
    rouge2_score = temp_rouge_score["rouge2"]
    rougeL_score = temp_rouge_score["rougeL"]

    # Calculate SacreBLEU and WER scores
    sacre_bleu_score = sacrebleu.compute(predictions=[output_text], references=[[target_sentence]])
    sacre_bleu_score = sacre_bleu_score["score"]
    wer_score = calculate_wer(target_sentence, output_text)

    # Calculate BERTScore metrics
    results = bertscore.compute(predictions=[output_text], references=[target_sentence], lang="en")

    precision_score = results["precision"]
    recall_score = results["recall"]
    f1_score = results["f1"]

    if isinstance(precision_score, list):
        precision_score = sum(precision_score) / len(precision_score)
    if isinstance(recall_score, list):
        recall_score = sum(recall_score) / len(recall_score)
    if isinstance(f1_score, list):
        f1_score = sum(f1_score) / len(f1_score)

    # Accumulate scores
    total_bleu_score += bleu_score

    total_rouge1_score += rouge1_score
    total_rouge2_score += rouge2_score
    total_rougeL_score += rougeL_score

    total_sacre_bleu_score += sacre_bleu_score
    total_wer_score += wer_score

    total_precision += precision_score
    total_recall += recall_score
    total_f1 += f1_score

    # Store individual scores for each sentence
    bleu_scores.append(bleu_score)
    rouge1_scores.append(rouge1_score)
    rouge2_scores.append(rouge2_score)
    rougeL_scores.append(rougeL_score)

    sacre_bleu_scores.append(sacre_bleu_score)
    wer_scores.append(wer_score)

    precision_scores.append(precision_score)
    recall_scores.append(recall_score)
    f1_scores.append(f1_score)


# Calculate average scores
avg_bleu_score = total_bleu_score / len(test_loader)
avg_sacre_bleu_score = total_sacre_bleu_score / len(test_loader)
avg_wer_score = total_wer_score / len(test_loader)

avg_rouge1_score = total_rouge1_score / len(test_loader)
avg_rouge2_score = total_rouge2_score / len(test_loader)
avg_rougeL_score = total_rougeL_score / len(test_loader)

avg_PR_score = total_precision / len(test_loader)
avg_RE_score = total_recall / len(test_loader)
avg_F1_score = total_f1 / len(test_loader)

# Print average scores
print()
print(f"Average BLEU Score: {(avg_bleu_score * 100):.4f}")
print(f"Average SacreBLEU Score: {avg_sacre_bleu_score:.4f}")
print(f"Average Word Error Rate (WER): {avg_wer_score}")

print(f"Average Rouge-1 Score: {avg_rouge1_score}")
print(f"Average Rouge-2 Score: {avg_rouge2_score}")
print(f"Average Rouge-L Score: {avg_rougeL_score}")

print(f"Average Precision: {avg_PR_score}")
print(f"Average Recall: {avg_RE_score}")
print(f"Average F1-Score: {avg_F1_score}")

print("evaluation ends!")

# **Run Inference**

In [ ]:
class CheckInferenceData(torch.utils.data.Dataset):
    def __init__(self, source_encodings, mask_encodings):
        self.source_encodings = source_encodings
        self.mask_encodings = mask_encodings

    def __getitem__(self, idx):
        input_ids = torch.tensor(self.source_encodings[idx]).squeeze()
        attention_mask = torch.tensor(self.mask_encodings[idx]).squeeze()
        return input_ids, attention_mask
    def __len__(self):
        return len(self.source_encodings)

In [ ]:
test_text_source = ['They currently working on a new project, but they attending a conference next week, and after that, they a well-deserved vacation.']
test_text_source_encoding = tokenizer(test_text_source, truncation=True, padding=True)

demo_text = CheckInferenceData(test_text_source_encoding['input_ids'], test_text_source_encoding['attention_mask'])
demo_loader = DataLoader(demo_text, batch_size=1, shuffle=False)

for input_ids_demo, attention_mask_demo in demo_loader:
    
    input_ids_demo = input_ids_demo.to(device)
    attention_mask_demo = attention_mask_demo.to(device)
    predictions_demo = model.generate(input_ids=input_ids_demo, attention_mask=attention_mask_demo)
    demo_prediction = [tokenizer.decode(token, skip_special_tokens=True) for token in predictions_demo]

In [ ]:
demo_prediction